# Task 3 — Analytics Charts
**BMIT3003 Data Warehouse Technology · Domain A (XS): Sales & Product**

Run every cell top to bottom. The last cell zips the PNGs and downloads them.

Eleven charts, each tied to one finding in report section 4.3. All figures are
transcribed from the output of `Task 3/task3_xs_reports.sql` — nothing is invented here.


In [ ]:
# =============================================================================
#  Task 3 - Analytics Charts        BMIT3003 Data Warehouse Technology
#  Domain A (XS): Sales & Product
#
#  Paste into Google Colab. Run top to bottom. The last cell zips every PNG
#  and downloads them.
#
#  Produces the ELEVEN charts used in report section 4.3. Others generated
#  during analysis were dropped: they duplicated an argument already made or
#  worked better as a small table in the text. The data cell is kept complete
#  so it still records every exhibit result.
#
#  Charts 01 and 09 were added after the growth decomposition (Exhibit 1.7)
#  and the period concentration split (Exhibits 2.7 / 2.7b) were introduced.
#  Both illustrate a headline finding that previously had no figure behind it.
#
#  All figures are transcribed from the warehouse output of
#  Task 3/task3_xs_reports.sql - no data is invented here.
#
#  THREE RULES BAKED INTO THESE CHARTS
#    1. 2026 is January to August only. Where it appears it is hatched and
#       labelled; it is excluded from every year-on-year calculation.
#    2. Chart 14 exists to show a DISCONTINUITY, not a trend. The 2024
#       boundary is where the reconstructed history meets the original
#       operational records.
#    3. Colours come from a CVD-validated categorical palette and are assigned
#       in fixed order. Never more than three series on one plot.
# =============================================================================


In [ ]:
# %% ===========================  CELL 1 - SETUP  =============================
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
import os

os.makedirs("charts", exist_ok=True)

# --- Validated categorical palette (fixed slot order, never cycled) ----------
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"      # blue, orange, aqua
RED, BLUE  = "#e34948", "#2a78d6"                  # diverging pair
INK        = "#1a1a19"
INK_SOFT   = "#52514e"
INK_MUTE   = "#8a8985"
GRID       = "#e6e5e1"
SURFACE    = "#ffffff"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,                 # print-quality for a Word report
    "savefig.bbox": "tight",
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "font.family": ["DejaVu Sans"],
    "font.size": 10,
    "axes.edgecolor": GRID,
    "axes.linewidth": 0.8,
    "axes.labelcolor": INK_SOFT,
    "axes.titlesize": 12.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": INK,
    "axes.titlepad": 14,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
})

def clean(ax, xgrid=False, ygrid=True):
    """Recessive axes: drop the box, keep one grid direction."""
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.xaxis.grid(xgrid)
    ax.yaxis.grid(ygrid)
    ax.tick_params(length=0)

def subtitle(ax, text):
    """Call AFTER set_title. Re-pads the title so the two never collide."""
    ax.set_title(ax.get_title(), pad=30)
    ax.text(0, 1.012, text, transform=ax.transAxes, fontsize=9.5,
            color=INK_MUTE, ha="left", va="bottom")

rm = FuncFormatter(lambda v, p: f"{v:,.0f}")
pct = FuncFormatter(lambda v, p: f"{v:.0f}%")

def save(fig, name):
    fig.savefig(f"charts/{name}.png", facecolor=SURFACE)
    print("saved charts/" + name + ".png")

print("Setup complete.")


In [ ]:
# %% ===========================  CELL 2 - DATA  ==============================
# ---- Exhibit 1.1 / 1.2 : yearly headline ------------------------------------
year        = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
orders      = [250, 330, 430, 540, 330, 450, 680, 900, 1100, 1350, 1002]
revenue     = [14337, 19155, 26101, 32144, 19640, 28213, 47682, 73699, 93023, 124891, 81865]
categories  = [6, 7, 8, 9, 10, 10, 11, 12, 12, 12, 12]
items_sold  = [28, 32, 36, 39, 43, 47, 49, 54, 54, 54, 54]
avg_basket  = [57.35, 58.05, 60.70, 59.53, 59.51, 62.70, 70.12, 81.89, 84.57, 92.51, 81.70]
yoy_pct     = [None, 33.6, 36.3, 23.2, -38.9, 43.7, 69.0, 54.6, 26.2, 34.3, None]
customers   = [103, 166, 233, 309, 236, 319, 451, 592, 645, 742, 621]
branches    = [4, 6, 7, 9, 9, 10, 11, 12, 12, 12, 12]
orders_per_cust = [2.43, 1.99, 1.85, 1.75, 1.40, 1.41, 1.51, 1.52, 1.71, 1.82, 1.61]

PART_YEAR = 2026   # January-August only

# ---- Exhibit 1.7 : growth decomposition, 2016 -> 2025 -----------------------
# Revenue is EXACTLY customers x orders-per-customer x basket, so these three
# multiples reconcile to the revenue multiple with nothing left over.
decomp = pd.DataFrame([
 ("Active customers",   103,   742,    742/103),
 ("Orders per customer", 2.43,  1.82,  1.82/2.43),
 ("Average basket (RM)", 57.35, 92.51, 92.51/57.35),
], columns=["component", "v2016", "v2025", "multiple"])
REV_MULT = 124891 / 14337
assert abs(decomp["multiple"].prod() - REV_MULT) < 0.02, "decomposition must reconcile"

# ---- Exhibit 1.3 : revenue by category by year ------------------------------
cat_rows = [
 (2016,"Bakery",1352),(2016,"Beverages",1772),(2016,"Dairy and Eggs",2960),
 (2016,"Frozen Food",3690),(2016,"Rice and Noodles",2487),(2016,"Snacks",2076),
 (2017,"Bakery",1570),(2017,"Beverages",2396),(2017,"Cooking Essentials",3087),
 (2017,"Dairy and Eggs",3942),(2017,"Frozen Food",3602),(2017,"Rice and Noodles",2317),
 (2017,"Snacks",2242),
 (2018,"Bakery",1880),(2018,"Beverages",2704),(2018,"Canned Food",1811),
 (2018,"Cooking Essentials",4137),(2018,"Dairy and Eggs",4165),(2018,"Frozen Food",5630),
 (2018,"Rice and Noodles",3153),(2018,"Snacks",2621),
 (2019,"Bakery",2171),(2019,"Beverages",3186),(2019,"Canned Food",3132),
 (2019,"Cooking Essentials",3846),(2019,"Dairy and Eggs",4876),(2019,"Frozen Food",6213),
 (2019,"Personal Care",2023),(2019,"Rice and Noodles",3875),(2019,"Snacks",2821),
 (2020,"Bakery",1290),(2020,"Beverages",1540),(2020,"Canned Food",1729),
 (2020,"Cooking Essentials",2473),(2020,"Dairy and Eggs",2602),(2020,"Frozen Food",3074),
 (2020,"Household Cleaning",351),(2020,"Personal Care",3039),
 (2020,"Rice and Noodles",1956),(2020,"Snacks",1586),
 (2021,"Bakery",1333),(2021,"Beverages",1999),(2021,"Canned Food",2480),
 (2021,"Cooking Essentials",3553),(2021,"Dairy and Eggs",3533),(2021,"Frozen Food",4429),
 (2021,"Household Cleaning",3132),(2021,"Personal Care",3113),
 (2021,"Rice and Noodles",2595),(2021,"Snacks",2045),
 (2022,"Baby Products",9695),(2022,"Bakery",2260),(2022,"Beverages",2604),
 (2022,"Canned Food",3196),(2022,"Cooking Essentials",4868),(2022,"Dairy and Eggs",3704),
 (2022,"Frozen Food",5671),(2022,"Household Cleaning",4518),(2022,"Personal Care",5064),
 (2022,"Rice and Noodles",3517),(2022,"Snacks",2586),
 (2023,"Baby Products",15237),(2023,"Bakery",2598),(2023,"Beverages",3598),
 (2023,"Canned Food",4997),(2023,"Cooking Essentials",5807),(2023,"Dairy and Eggs",5505),
 (2023,"Frozen Food",6662),(2023,"Household Cleaning",4512),(2023,"Personal Care",7385),
 (2023,"Pet Care",10566),(2023,"Rice and Noodles",4165),(2023,"Snacks",2667),
 (2024,"Baby Products",18182),(2024,"Bakery",3611),(2024,"Beverages",5599),
 (2024,"Canned Food",5784),(2024,"Cooking Essentials",7751),(2024,"Dairy and Eggs",8091),
 (2024,"Frozen Food",9495),(2024,"Household Cleaning",6982),(2024,"Personal Care",7408),
 (2024,"Pet Care",11440),(2024,"Rice and Noodles",5134),(2024,"Snacks",3547),
 (2025,"Baby Products",27619),(2025,"Bakery",4546),(2025,"Beverages",5860),
 (2025,"Canned Food",7066),(2025,"Cooking Essentials",8454),(2025,"Dairy and Eggs",9392),
 (2025,"Frozen Food",11385),(2025,"Household Cleaning",8903),(2025,"Personal Care",9667),
 (2025,"Pet Care",18969),(2025,"Rice and Noodles",8724),(2025,"Snacks",4305),
 (2026,"Baby Products",16073),(2026,"Bakery",3099),(2026,"Beverages",4033),
 (2026,"Canned Food",4065),(2026,"Cooking Essentials",6538),(2026,"Dairy and Eggs",7466),
 (2026,"Frozen Food",8135),(2026,"Household Cleaning",5814),(2026,"Personal Care",6580),
 (2026,"Pet Care",11917),(2026,"Rice and Noodles",4697),(2026,"Snacks",3448),
]
cat = pd.DataFrame(cat_rows, columns=["year", "category", "revenue"])

# ---- Exhibit 1.4 : category entry ------------------------------------------
entry = pd.DataFrame([
 ("Frozen Food",2016,5,67987,12.1),("Dairy and Eggs",2016,4,56235,10.0),
 ("Rice and Noodles",2016,4,42621,7.6),("Beverages",2016,5,35290,6.3),
 ("Snacks",2016,5,29943,5.3),("Bakery",2016,5,25708,4.6),
 ("Cooking Essentials",2017,5,50514,9.0),("Canned Food",2018,5,34259,6.1),
 ("Personal Care",2019,4,44281,7.9),("Household Cleaning",2020,5,34212,6.1),
 ("Baby Products",2022,4,86807,15.5),("Pet Care",2023,4,52892,9.4),
], columns=["category","first_year","items","lifetime_revenue","pct_of_total"])

# Cohort = the range-expansion story, and it keeps us to three series
def cohort(y):
    if y == 2016: return "Founding six (2016)"
    if y <= 2020: return "Added 2017-2020"
    return "Late entrants (2022-23)"
entry["cohort"] = entry["first_year"].apply(cohort)
COHORTS = ["Founding six (2016)", "Added 2017-2020", "Late entrants (2022-23)"]
COHORT_C = {COHORTS[0]: C1, COHORTS[1]: C2, COHORTS[2]: C3}
cat = cat.merge(entry[["category","cohort"]], on="category", how="left")

# ---- Exhibit 1.5 / 1.6 : seasonality ---------------------------------------
quarters   = ["Q1", "Q2", "Q3", "Q4"]
q_orders   = [1709, 1544, 1455, 1652]
q_index    = [107, 97, 92, 104]

q1_share = pd.DataFrame([
 ("Pet Care",29.5),("Dairy and Eggs",27.6),("Canned Food",27.4),("Snacks",26.1),
 ("Frozen Food",26.1),("Household Cleaning",25.7),("Bakery",25.7),
 ("Rice and Noodles",25.1),("Cooking Essentials",25.0),("Personal Care",24.6),
 ("Beverages",24.5),("Baby Products",20.9),
], columns=["category","q1_pct"])
STORE_Q1 = 1709 / (1709+1544+1455+1652) * 100      # store-wide Q1 share

# ---- Exhibit 2.1 / 2.3 : suppliers -----------------------------------------
sup = pd.DataFrame([
 ("LittleStar Baby Products",86807,15.5,"Baby Products",3094,82,89.9,-0.83),
 ("Polar Frozen Foods",      67987,12.1,"Frozen Food",  6831,219,198.4, 1.46),
 ("Fresh Dairy Farm",        56235,10.0,"Dairy and Eggs",5457,163,158.5, 0.36),
 ("PetJoy Trading",          52892, 9.4,"Pet Care",     2836, 99, 82.4, 1.83),
 ("Selera Cooking Products", 50514, 9.0,"Cooking Ess.", 6182,166,179.6,-1.01),
 ("CarePlus Consumer Goods", 44281, 7.9,"Personal Care",3891,103,113.0,-0.94),
 ("Padi Emas Rice Mills",    42621, 7.6,"Rice & Noodles",5356,152,155.6,-0.29),
 ("Sunshine Beverages",      35290, 6.3,"Beverages",    6679,165,194.0,-2.08),
 ("Ocean Canned Food",       34259, 6.1,"Canned Food",  5829,177,169.3, 0.59),
 ("KleenHome Supplies",      34212, 6.1,"Household Cl.",4536,172,131.7, 3.51),
 ("Snacko Food Industries",  29943, 5.3,"Snacks",       5869,148,170.5,-1.72),
 ("Gardenview Bakery",       25708, 4.6,"Bakery",       6652,190,193.2,-0.23),
], columns=["supplier","revenue","pct","category","units_sold",
            "returned","expected","z"])
sup["cum_pct"] = sup["pct"].cumsum()
# A return line takes back 1-3 units, so returns arrive in clusters. The plain
# Poisson SD assumes single units and therefore understates true variance by
# about 1.5x. Dividing by that factor is the honest z.
CLUSTER_ADJ = 1.53
sup["z_adj"] = sup["z"] / CLUSTER_ADJ

# ---- Exhibit 2.7 / 2.7b : concentration by period ---------------------------
# Revenue concentration and STRUCTURAL exposure move in opposite directions:
# a wider range spreads revenue over more suppliers while creating more
# categories that are each still single-sourced. 2026 excluded (part year).
period = pd.DataFrame([
 ("2016-2020", sum(revenue[0:5]), 10, 19.9, 61.1, "Polar Frozen Foods"),
 ("2021-2025", sum(revenue[5:10]), 12, 19.2, 49.5, "LittleStar Baby Products"),
], columns=["period","revenue","suppliers","largest_pct","top4_pct","largest"])

# ---- Exhibit 2.4 : return reasons ------------------------------------------
reasons = pd.DataFrame([
 ("Wrong Item","Fulfilment",243,25.7),("Broken","Product Quality",235,24.9),
 ("Expired","Product Quality",233,24.7),("Missing","Fulfilment",233,24.7),
], columns=["reason","group","lines","pct"])

# ---- Exhibit 3.1 / 3.2 / 3.3 / 3.4 : channel -------------------------------
online_pct = [17.6, 20.6, 27.9, 30.4, 34.5, 37.1, 42.6, 42.7, 42.8, 44.9, 50.5]

region = pd.DataFrame([
 ("Northern",876,48.5,89.46),("East Malaysia",287,46.0,83.86),
 ("Central",1403,45.9,87.47),("East Coast",299,44.8,85.37),
 ("Southern",587,42.2,83.62),
], columns=["region","orders","online_pct","avg_basket"])

tier = pd.DataFrame([
 ("VIP",2218,44.9,76.63),("Normal",2776,43.1,77.27),("Non-Member",2368,31.4,74.45),
], columns=["tier","orders","online_pct","avg_basket"])

basket = pd.DataFrame([
 (2022,70.66,69.72),(2023,76.55,85.86),(2024,84.35,84.73),
 (2025,92.02,92.91),(2026,81.54,81.87),
], columns=["year","online","walkin"])

# ---- Confounded series (charted to SHOW the break, not a trend) -------------
return_pct = [2.49, 2.80, 1.77, 2.45, 2.05, 2.57, 2.60, 2.83, 3.21, 3.86, 2.70]
lead_days  = [3.70, 3.97, 4.18, 3.74, 4.46, 4.39, 4.02, 3.99, 2.60, 2.87, 3.08]
cancel_pct = [6.90, 8.82, 4.69, 12.37, 14.29, 11.58, 12.24, 9.13, 5.73, 5.15, 8.83]
BOUNDARY   = 2024

print("Data loaded:", len(cat), "category-year rows,", len(sup), "suppliers.")


In [ ]:
# %% ==========================  CELL 3 - REPORT 1: RANGE EXPANSION AND SEASONALITY  ==========================
# Charts 01, 02, 03, 05 - the four used in section 4.3.1.

# --- Chart 01: WHO grew, and which driver actually did the work -------------
# Left panel is the acquisition curve (the WHO). Right panel is Exhibit 1.7:
# the three multiples that reconcile to the 8.71x revenue growth. Anything
# left of the 1.0 line worked AGAINST growth.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4),
                         gridspec_kw={"width_ratios": [1.25, 1]})

ax = axes[0]
pre  = [i for i, y in enumerate(year) if y <= 2025]
ax.plot([year[i] for i in pre], [customers[i] for i in pre], color=C1, lw=2.6,
        marker="o", ms=6, mfc=SURFACE, mew=2.2, zorder=4)
ax.plot([2025, 2026], [customers[9], customers[10]], color=C1, lw=2.6, ls=":",
        marker="o", ms=6, mfc=SURFACE, mew=2.2, alpha=0.55, zorder=4)
ax.set_title("103 customers in 2016, 742 by 2025", fontsize=11.5)
ax.set_ylabel("Active customers"); ax.set_xticks(year[::2]); clean(ax)
ax.annotate("2020: base falls\nfor the only time", xy=(2020, 236),
            xytext=(2016.4, 470), fontsize=8.5, color=INK_SOFT,
            arrowprops=dict(arrowstyle="->", color=INK_MUTE, lw=1))
ax.text(2026, customers[10] - 55, "2026\npart year", ha="center", va="top",
        fontsize=8, color=INK_MUTE)

ax = axes[1]
d = decomp.iloc[::-1]
cols = [BLUE if m >= 1 else RED for m in d["multiple"]]
ax.barh(d["component"], d["multiple"], height=0.6, color=cols, zorder=3)
ax.axvline(1.0, color=INK_SOFT, lw=1.2, zorder=4)
ax.text(1.03, 2.52, "no change", fontsize=8.5, color=INK_SOFT)
for i, (m, c) in enumerate(zip(d["multiple"], d["component"])):
    ax.text(m + 0.16, i, f"x{m:.2f}", va="center", fontsize=10,
            weight="bold", color=INK)
ax.set_title("Customer growth did the work", fontsize=11.5)
ax.set_xlabel("Growth multiple, 2016 to 2025"); ax.set_xlim(0, 8.6)
clean(ax, xgrid=True, ygrid=False)
fig.tight_layout()
fig.suptitle("Revenue grew 8.71x - acquisition x7.20, basket x1.61, frequency x0.75",
             fontsize=12.5, weight="bold", color=INK, y=1.10)
fig.text(0.5, 1.035, "The three multiples reconcile exactly to the revenue growth. "
                     "Order frequency FELL, offsetting a quarter of the gain.",
         fontsize=9.5, color=INK_MUTE, ha="center", va="bottom")
save(fig, "01_growth_drivers"); plt.show()


# --- Year-on-year: a diverging bar, which is what polarity data wants --------
yy = [(y, v) for y, v in zip(year, yoy_pct) if v is not None]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar([y for y, _ in yy], [v for _, v in yy], width=0.66, zorder=3,
       color=[BLUE if v > 0 else RED for _, v in yy])
ax.axhline(0, color=INK_SOFT, lw=1)
ax.set_title("A 38.9% collapse in 2020, then two years to recover")
subtitle(ax, "Year-on-year change in net revenue. 2026 excluded: a part year carries no YoY.")
ax.yaxis.set_major_formatter(pct); ax.set_ylabel("YoY change")
ax.set_xticks([y for y, _ in yy]); clean(ax)
for y, v in yy:
    ax.text(y, v + (3 if v > 0 else -6), f"{v:+.1f}%", ha="center",
            fontsize=8.5, color=INK_SOFT)
save(fig, "02_yoy_growth"); plt.show()

piv = (cat.pivot_table(index="year", columns="cohort", values="revenue",
                       aggfunc="sum").fillna(0)[COHORTS])
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.stackplot(piv.index, [piv[c] for c in COHORTS],
             colors=[COHORT_C[c] for c in COHORTS], labels=COHORTS,
             edgecolor=SURFACE, linewidth=2)          # 2px surface gap between fills
ax.set_title("Two late categories drive a quarter of all revenue")
subtitle(ax, "Net revenue by the year each category entered the range. RM.")
ax.yaxis.set_major_formatter(rm); ax.set_ylabel("Net revenue (RM)")
ax.set_xticks(year); ax.set_xlim(2016, 2026); clean(ax)
ax.axvline(PART_YEAR, color=INK_MUTE, lw=1, ls=":", zorder=5)
ax.text(2025.95, ax.get_ylim()[1]*0.97, "2026 part year ", ha="right", va="top",
        fontsize=8.5, color=INK_MUTE)
ax.legend(loc="upper left", ncol=1)
save(fig, "03_cohort_stack"); plt.show()

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.bar(quarters, q_index, color=C1, width=0.6, zorder=3)
ax.axhline(100, color=INK_SOFT, lw=1.2, ls="--", zorder=4)
ax.text(3.42, 100.4, "average quarter", ha="right", fontsize=8.5, color=INK_SOFT)
ax.set_title("Q1 and Q4 carry the year")
subtitle(ax, "Orders indexed to the average quarter = 100. 2026 excluded.")
ax.set_ylim(85, 112); ax.set_ylabel("Order index"); clean(ax)
for q, v in zip(quarters, q_index):
    ax.text(q, v + 0.8, str(v), ha="center", fontsize=10, color=INK, weight="bold")
save(fig, "05_quarterly_index"); plt.show()


In [ ]:
# %% ==========================  CELL 4 - REPORT 2: SUPPLIER CONCENTRATION AND RETURNS  ==========================
# Charts 07, 08 - the two used in section 4.3.2.

s = sup.sort_values("revenue")
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.barh(s["supplier"], s["revenue"], color=C1, height=0.62, zorder=3)
ax.set_title("Every supplier is the sole source of exactly one category")
subtitle(ax, "Lifetime net revenue, RM. The category each supplier solely supplies is named.")
ax.xaxis.set_major_formatter(rm); ax.set_xlabel("Lifetime net revenue (RM)")
clean(ax, xgrid=True, ygrid=False); ax.set_xlim(0, 112000)
for i, r in enumerate(s.itertuples()):
    ax.text(r.revenue + 1600, i, f"{r.pct}%  ·  {r.category}",
            va="center", fontsize=8.5, color=INK_SOFT)
save(fig, "07_supplier_pareto"); plt.show()

# --- THE key exhibit: is any supplier a real quality outlier? ----------------
sz = sup.sort_values("z")
ypos = np.arange(len(sz))
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.axvspan(-2.87, 2.87, color=C1, alpha=0.07, zorder=1)
ax.axvspan(-2, 2, color=C1, alpha=0.10, zorder=1)
ax.axvline(0, color=INK_SOFT, lw=1.2, zorder=2)
for x in (-2, 2):
    ax.axvline(x, color=INK_MUTE, lw=1, ls="--", zorder=2)
ax.hlines(ypos, sz["z_adj"], sz["z"], color=GRID, lw=1.6, zorder=3)
ax.scatter(sz["z"], ypos, s=95, color=C2, zorder=5, label="Raw z (Poisson)")
ax.scatter(sz["z_adj"], ypos, s=95, color=C1, zorder=5,
           label="Adjusted for return clustering")
ax.set_yticks(ypos); ax.set_yticklabels(sz["supplier"])
ax.set_title("No supplier's return rate departs far enough to act on")
subtitle(ax, "Deviation from expected returns, in standard deviations. "
             "Shaded bands: +/-2 and +/-2.87 (Bonferroni, 12 tests).")
ax.set_xlabel("z-score"); ax.set_xlim(-4.6, 4.6)
clean(ax, xgrid=True, ygrid=False)
ax.legend(loc="lower right")
ax.text(-4.45, len(sz) - 1.15,
        "KleenHome is the only supplier outside +/-2 on the raw score.\n"
        "Adjusted for clustering it falls to 2.29 - inside the band that\n"
        "12 simultaneous comparisons require.",
        fontsize=8.5, color=INK_SOFT, va="top", linespacing=1.5)
save(fig, "08_supplier_zscore"); plt.show()


# --- Chart 09: WHEN - has the exposure grown or eased? ----------------------
# The counterintuitive one. Revenue concentration FELL (top-4 61.1% -> 49.5%)
# while structural exposure ROSE (10 -> 12 sole-source suppliers), because the
# dispersion came from opening new categories, not from second-sourcing old
# ones. The largest single dependency barely moved; it only changed identity.
fig, ax = plt.subplots(figsize=(8.4, 4.4))
x = np.arange(len(period)); w = 0.32
ax.bar(x - w/2, period["top4_pct"], w, color=C1, zorder=3,
       label="Top-four suppliers' share of revenue")
ax.bar(x + w/2, period["largest_pct"], w, color=C2, zorder=3,
       label="Largest single supplier's share")
for i, r in period.iterrows():
    ax.text(i - w/2, r.top4_pct + 1.4, f"{r.top4_pct}%", ha="center",
            fontsize=10, weight="bold", color=INK)
    ax.text(i + w/2, r.largest_pct + 1.4, f"{r.largest_pct}%", ha="center",
            fontsize=10, weight="bold", color=INK)
    ax.text(i, -6.4, f"{r.suppliers} sole-source suppliers",
            ha="center", fontsize=9.5, color=RED if r.suppliers == 12 else INK_SOFT,
            weight="bold" if r.suppliers == 12 else "normal")
    ax.text(i + w/2, r.largest_pct - 3.2, r.largest.split()[0], ha="center",
            fontsize=8.5, color=SURFACE, weight="bold")
ax.set_title("Revenue spread out - but the number of single points of failure rose")
subtitle(ax, "Two five-year periods. 2026 excluded as a part year.")
ax.set_xticks(x); ax.set_xticklabels(period["period"], fontsize=10.5)
ax.set_ylim(0, 78); ax.yaxis.set_major_formatter(pct)
ax.set_ylabel("Share of period revenue")
clean(ax); ax.legend(loc="upper right")
ax.annotate("", xy=(1 - w/2, 53.6), xytext=(0 - w/2, 65.0),
            arrowprops=dict(arrowstyle="->", color=INK_MUTE, lw=1.4, ls="--"))
ax.text(0.5, 60.6, "-11.6 pts", ha="center", fontsize=9.5, color=INK_SOFT,
        bbox=dict(fc=SURFACE, ec="none", pad=1.5))
save(fig, "09_concentration_period"); plt.show()


In [ ]:
# %% ==========================  CELL 5 - REPORT 3: CHANNEL MIGRATION  ==========================
# Charts 10, 11, 12 - the three used in section 4.3.3.

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(year, online_pct, color=C1, lw=2.4, marker="o", ms=7,
        mfc=SURFACE, mew=2.2, zorder=4)
ax.axhline(50, color=INK_MUTE, lw=1, ls="--", zorder=2)
ax.text(2016, 50.7, "half of all orders", fontsize=8.5, color=INK_SOFT)
ax.set_title("Online crossed half of all orders in 2026")
subtitle(ax, "Online share of orders. Share is unaffected by 2026 being a part year.")
ax.yaxis.set_major_formatter(pct); ax.set_ylabel("Online share of orders")
ax.set_xticks(year); ax.set_ylim(10, 58); clean(ax)
for y, v in [(2016, 17.6), (2026, 50.5)]:
    ax.annotate(f"{v}%", xy=(y, v), xytext=(0, 13), textcoords="offset points",
                ha="center", fontsize=10, weight="bold", color=INK)
save(fig, "10_online_share"); plt.show()

# --- Membership: two measures, so two panels. Never a dual axis. -------------
t = tier.sort_values("online_pct")
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))
axes[0].barh(t["tier"], t["online_pct"],
             color=[C2 if x == "Non-Member" else C1 for x in t["tier"]],
             height=0.6, zorder=3)
axes[0].set_title("Members buy online", fontsize=11.5)
axes[0].xaxis.set_major_formatter(pct); axes[0].set_xlim(0, 55)
clean(axes[0], xgrid=True, ygrid=False)
for i, row in enumerate(t.itertuples()):
    axes[0].text(row.online_pct + 1, i, f"{row.online_pct}%", va="center",
                 fontsize=9, color=INK_SOFT)

axes[1].barh(t["tier"], t["avg_basket"],
             color=[C2 if x == "Non-Member" else C1 for x in t["tier"]],
             height=0.6, zorder=3)
axes[1].set_title("...but do not spend more", fontsize=11.5)
axes[1].set_xlim(0, 95); clean(axes[1], xgrid=True, ygrid=False)
for i, row in enumerate(t.itertuples()):
    axes[1].text(row.avg_basket + 1.6, i, f"RM {row.avg_basket:.2f}", va="center",
                 fontsize=9, color=INK_SOFT)
fig.suptitle("Membership drives channel, not spend", fontsize=12.5,
             fontweight="bold", color=INK, y=1.06, x=0.02, ha="left")
save(fig, "11_membership_tier"); plt.show()

# --- Online vs walk-in basket: the lines sit on top of each other ------------
fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.plot(basket["year"], basket["online"], color=C1, lw=2.4, marker="o", ms=7,
        mfc=SURFACE, mew=2.2, label="Online", zorder=4)
ax.plot(basket["year"], basket["walkin"], color=C2, lw=2.4, marker="s", ms=7,
        mfc=SURFACE, mew=2.2, label="Walk-in", zorder=4)
ax.set_title("The migration substitutes, it does not grow the basket")
subtitle(ax, "Average basket value by channel, RM. The two lines track within RM 1.")
ax.set_ylabel("Average basket (RM)"); ax.set_xticks(basket["year"])
clean(ax); ax.legend(loc="lower right")
save(fig, "12_basket_by_channel"); plt.show()


In [ ]:
# %% ==========================  CELL 6 - LIMITATIONS: THE 2024 DISCONTINUITY  ==========================
# Chart 14 - used in section 4.3.4. This chart exists to show a
# DISCONTINUITY, not a trend: the step sits exactly where the reconstructed
# history meets the original operational records. Do not describe it as an
# improvement or a deterioration.

fig, ax = plt.subplots(figsize=(9, 4.2))
pre  = [i for i, y in enumerate(year) if y <  BOUNDARY]
post = [i for i, y in enumerate(year) if y >= BOUNDARY]
ax.plot([year[i] for i in pre], [return_pct[i] for i in pre], color=C1, lw=2.4,
        marker="o", ms=7, mfc=SURFACE, mew=2.2, label="Reconstructed history", zorder=4)
ax.plot([year[i] for i in post], [return_pct[i] for i in post], color=C2, lw=2.4,
        marker="o", ms=7, mfc=SURFACE, mew=2.2, label="Original records (blended)", zorder=4)
ax.axvline(BOUNDARY - 0.5, color=INK_SOFT, lw=1.4, ls="--", zorder=3)
ax.text(BOUNDARY - 0.42, ax.get_ylim()[1], " dataset boundary", va="top",
        fontsize=9, color=INK_SOFT)
ax.set_title("Return rate does not rise - the datasets change")
subtitle(ax, "The step sits exactly where the two data sources meet, not across the series.")
ax.set_ylabel("Units returned as % of units sold"); ax.set_xticks(year); clean(ax)
ax.legend(loc="upper left")
save(fig, "14_confound_return_rate"); plt.show()


In [ ]:
# %% ===================  CELL 7 - DOWNLOAD EVERYTHING  =======================
import shutil
shutil.make_archive("task3_charts", "zip", "charts")
print("\nCharts written:")
for f in sorted(os.listdir("charts")):
    print("  ", f)

try:
    from google.colab import files
    files.download("task3_charts.zip")
except Exception as e:
    print("\nNot running in Colab - the PNGs are in ./charts/  (", e, ")")
